### Импорт библиотек

In [2]:
import os
import pandas as pd
import numpy as np
from pathlib import Path
import re
from datetime import date, timedelta, datetime
import locale
from decimal import Decimal, ROUND_UP
import warnings
from io import BytesIO
from pandas.tseries.offsets import MonthEnd
from dateutil.relativedelta import relativedelta

warnings.filterwarnings('ignore')

locale.setlocale(locale.LC_TIME, 'ru')
pd.set_option('display.max_columns', None)

pd.options.display.float_format ='{:.2f}'.format

### Незаконтрактованные с остатком > 500 т.р.

#### Обработка исходников

In [3]:
num_old = '31.12'
dt_old = datetime.strptime('31.12.2025', '%d.%m.%Y')

num_new = '04.05'
dt_new = datetime.strptime('04.05.2026', '%d.%m.%Y')

In [4]:
yesterday = dt_new - relativedelta(days=1)
one_year_later = dt_new + relativedelta(years=1) + relativedelta(months=1)
last_day_next_month = dt_new.replace(day=1) + relativedelta(months=2)

In [5]:
file_path = 'C:/Users/user078/Desktop/Отчеты 1-4/1 _ исполнение/Исполнение 1с текущее_2.xlsx'

In [6]:
skip_rows = 3
# Проверим так ли это
# Открываем файл и читаем строки до тех пор, пока не найдем заголовок
with pd.ExcelFile(file_path) as xls:
    # Предполагаем, что шапка не превышает 100 строк
    for i in range(100):
        df_temp = pd.read_excel(xls,
                                nrows=1,
                                skiprows=i)
        # Ищем название первого столбца
        if df_temp.columns[0] == 'Дата исполнения':
            print("break")
            skip_rows = i
            break
print("skip_rows _", skip_rows)

break
skip_rows _ 3


In [ ]:
isp = pd.read_excel(
    file_path,
    skiprows=skip_rows,
    skipfooter=1
    )
isp.head(2)

In [8]:
# заполним пропуски УТ, где это возможно
isp["Номенклатура.Код"] = (
    isp
    .groupby(["Торговая упаковка.Фирма производитель",
            "МНН",
            "Наименование полное",
            ])
    ["Номенклатура.Код"]
    .transform(lambda x: x.fillna(x.dropna().iloc[0]) if not x.dropna().empty else x)
)
isp['Номенклатура.Код'].isna().sum()

np.int64(29)

In [9]:
isp = isp.drop_duplicates()

In [ ]:
isp = isp[['Торговая упаковка.Фирма производитель', 'Номенклатура.Код', 'Наименование полное', 'Номер аукциона',
           'Количество осталось отгрузить в целом', 'Договор.Документ исполнение.Ответственный', 'Количество упаковок в заявке', 'Субъект', 'Заказчик в шапке', 'Цена ГК за упак. с НДС (нов.)']]
isp['Количество осталось отгрузить в целом'] = isp['Количество осталось отгрузить в целом'].fillna(0)
isp = isp.rename(columns={'Договор.Документ исполнение.Ответственный':'Ответственный'})
isp.head()

In [3]:
# Пример данных
demo_overstock_detail = pd.DataFrame([
    {
        "Производитель_анон": "Производитель_001",
        "Код_анон": "Код_001",
        "Наименование_анон": "Препарат_полный_001",
        "Номер аукциона_анон": "Аукцион_001",
        "Кол-во осталось отгрузить_анон": 0.0,
        "Ответственный": "М76",
        "Кол-во упаковок в заявке_анон": np.nan,
        "Субъект_анон": "Регион_001",
        "Заказчик_анон": "Заказчик_001",
        "Цена ГК за упак. с НДС_анон": 59.0,
    },
    {
        "Производитель_анон": "Производитель_001",
        "Код_анон": "Код_001",
        "Наименование_анон": "Препарат_полный_001",
        "Номер аукциона_анон": "Аукцион_002",
        "Кол-во осталось отгрузить_анон": 0.0,
        "Ответственный": "М542",
        "Кол-во упаковок в заявке_анон": np.nan,
        "Субъект_анон": "Регион_002",
        "Заказчик_анон": "Заказчик_002",
        "Цена ГК за упак. с НДС_анон": 51.96,
    },
    {
        "Производитель_анон": "Производитель_001",
        "Код_анон": "Код_002",
        "Наименование_анон": "Препарат_полный_002",
        "Номер аукциона_анон": "Аукцион_003",
        "Кол-во осталось отгрузить_анон": 100.0,
        "Ответственный": "М49",
        "Кол-во упаковок в заявке_анон": np.nan,
        "Субъект_анон": "Регион_003",
        "Заказчик_анон": "Заказчик_003",
        "Цена ГК за упак. с НДС_анон": 153.06,
    },
    {
        "Производитель_анон": "Производитель_001",
        "Код_анон": "Код_002",
        "Наименование_анон": "Препарат_полный_002",
        "Номер аукциона_анон": "Аукцион_004",
        "Кол-во осталось отгрузить_анон": 17.0,
        "Ответственный": "М76",
        "Кол-во упаковок в заявке_анон": np.nan,
        "Субъект_анон": "Регион_003",
        "Заказчик_анон": "Заказчик_004",
        "Цена ГК за упак. с НДС_анон": 136.0,
    },
    {
        "Производитель_анон": "Производитель_001",
        "Код_анон": "Код_002",
        "Наименование_анон": "Препарат_полный_002",
        "Номер аукциона_анон": "Аукцион_005",
        "Кол-во осталось отгрузить_анон": 1.0,
        "Ответственный": "М76",
        "Кол-во упаковок в заявке_анон": np.nan,
        "Субъект_анон": "Регион_004",
        "Заказчик_анон": "Заказчик_005",
        "Цена ГК за упак. с НДС_анон": 92.91,
    },
])

demo_overstock_detail

,Производитель_анон,Код_анон,Наименование_анон,Номер аукциона_анон,Кол-во осталось отгрузить_анон,Ответственный,Кол-во упаковок в заявке_анон,Субъект_анон,Заказчик_анон,Цена ГК за упак. с НДС_анон
0,Производитель_001,Код_001,Препарат_полный_001,Аукцион_001,0.00,М76,NaN,Регион_001,Заказчик_001,59.00
1,Производитель_001,Код_001,Препарат_полный_001,Аукцион_002,0.00,М542,NaN,Регион_002,Заказчик_002,51.96
2,Производитель_001,Код_002,Препарат_полный_002,Аукцион_003,100.00,М49,NaN,Регион_003,Заказчик_003,153.06
3,Производитель_001,Код_002,Препарат_полный_002,Аукцион_004,17.00,М76,NaN,Регион_003,Заказчик_004,136.00
4,Производитель_001,Код_002,Препарат_полный_002,Аукцион_005,1.00,М76,NaN,Регион_004,Заказчик_005,92.91


#### Восстановление УТ

In [ ]:
code_UT = pd.read_excel('//fs/АДМИНИСТРАЦИЯ/Сводные отчеты/Транскрипции/Ут_Востановление.xlsx')
print(code_UT.shape)
code_UT.head()

In [14]:
code_UT['chain'] = code_UT['Производитель'] + code_UT['Наименование полное']

In [15]:
code_UT = code_UT.rename(columns={'Производитель':'Торговая упаковка.Фирма производитель'})

In [16]:
isp = isp.merge(code_UT[['Торговая упаковка.Фирма производитель','Наименование полное','Код']], on=['Торговая упаковка.Фирма производитель','Наименование полное'], how='left')

In [ ]:
isp['Номенклатура.Код'] = isp['Номенклатура.Код'].fillna(isp['Код'])
isp.head()

#### Доступные остатки

In [18]:
file_path = 'C:/Users/user078/Desktop/Отчеты 1-4/1 _ исполнение/Доступные остатки_2.xlsx'

In [19]:
skip_rows = 6
# Проверим так ли это
# Открываем файл и читаем строки до тех пор, пока не найдем заголовок
with pd.ExcelFile(file_path) as xls:
    # Предполагаем, что шапка не превышает 100 строк
    for i in range(100):
        df_temp = pd.read_excel(xls,
                                nrows=1,
                                skiprows=i)
        # Ищем название первого столбца
        if df_temp.columns[0] == 'Номенклатура.Код':
            print("break")
            skip_rows = i
            break
print("skip_rows _", skip_rows)

break
skip_rows _ 6


In [ ]:
aviable_leftovers = pd.read_excel(
    file_path,
    skiprows=skip_rows,
    skipfooter=1
    )
aviable_leftovers.head(2)

In [ ]:
aviable_leftovers = aviable_leftovers.rename(columns={'Unnamed: 11': 'Доступно', 'Unnamed: 10': 'В резерве',
                                                      'Unnamed: 9': 'Отгружается'})
aviable_leftovers = aviable_leftovers.drop(0)
aviable_leftovers = aviable_leftovers[['Номенклатура.Код', 'Номенклатура', 'Доступно', 'В резерве', 'Отгружается']]
aviable_leftovers.head(2)

In [4]:
demo_stock_data = pd.DataFrame([
    {
        "Код_анон": "Код_101",
        "Номенклатура_анон": "Препарат_101",
        "Доступно": 400,
        "В резерве": np.nan,
        "Отгружается": np.nan,
    },
    {
        "Код_анон": "Код_102",
        "Номенклатура_анон": "Препарат_102",
        "Доступно": 625,
        "В резерве": np.nan,
        "Отгружается": np.nan,
    },
])

demo_stock_data

,Код_анон,Номенклатура_анон,Доступно,В резерве,Отгружается
0,Код_101,Препарат_101,400,NaN,NaN
1,Код_102,Препарат_102,625,NaN,NaN


In [22]:
aviable_leftovers[['Доступно', 'В резерве', 'Отгружается']] = aviable_leftovers[['Доступно', 'В резерве', 'Отгружается']].fillna(0)

In [23]:
aviable_leftovers['Доступно'] = aviable_leftovers['Доступно'] + aviable_leftovers['В резерве']

In [24]:
# conds = []
# for a, b in rules:
#     cond = (aviable_leftovers['Номенклатура'].str.contains(a) &
#             aviable_leftovers['Номенклатура'].str.contains(b))
#     conds.append(cond)

# # пример: объединить все условия
# final_cond = conds[0]
# for c in conds[1:]:
#     final_cond |= c

# aviable_leftovers = aviable_leftovers[final_cond]

In [ ]:
aviable_leftovers.sort_values(by='Номенклатура').head(3)

#### Транскрибирование менеджеров

In [ ]:
trans = pd.read_excel('Транскрипция для дашборда.xlsx', usecols=('A,B'))
trans.head(3)

In [ ]:
isp_trans = pd.merge(isp, trans, on='Ответственный', how='left').reset_index(drop=True)
isp_trans.head(3)

In [5]:
demo_contracts_detail = pd.DataFrame([
    {
        "Производитель_анон": "Производитель_001",
        "Код_анон": "Код_001",
        "Наименование_анон": "Препарат_полный_001",
        "Номер аукциона_анон": "Аукцион_001",
        "Кол-во осталось отгрузить_анон": 0.0,
        "Ответственный": "М76",
        "Кол-во упаковок в заявке_анон": np.nan,
        "Субъект_анон": "Регион_001",
        "Заказчик_анон": "Заказчик_001",
        "Цена ГК за упак. с НДС_анон": 59.0,
        "Код_анон_2": np.nan,
        "Менеджер_анон": "Менеджер_001",
    },
    {
        "Производитель_анон": "Производитель_001",
        "Код_анон": "Код_001",
        "Наименование_анон": "Препарат_полный_001",
        "Номер аукциона_анон": "Аукцион_002",
        "Кол-во осталось отгрузить_анон": 0.0,
        "Ответственный": "М542",
        "Кол-во упаковок в заявке_анон": np.nan,
        "Субъект_анон": "Регион_002",
        "Заказчик_анон": "Заказчик_002",
        "Цена ГК за упак. с НДС_анон": 51.96,
        "Код_анон_2": np.nan,
        "Менеджер_анон": "Менеджер_002",
    },
    {
        "Производитель_анон": "Производитель_001",
        "Код_анон": "Код_002",
        "Наименование_анон": "Препарат_полный_002",
        "Номер аукциона_анон": "Аукцион_003",
        "Кол-во осталось отгрузить_анон": 100.0,
        "Ответственный": "М49",
        "Кол-во упаковок в заявке_анон": np.nan,
        "Субъект_анон": "Регион_003",
        "Заказчик_анон": "Заказчик_003",
        "Цена ГК за упак. с НДС_анон": 153.06,
        "Код_анон_2": np.nan,
        "Менеджер_анон": "Менеджер_003",
    },
])

demo_contracts_detail

,Производитель_анон,Код_анон,Наименование_анон,Номер аукциона_анон,Кол-во осталось отгрузить_анон,Ответственный,Кол-во упаковок в заявке_анон,Субъект_анон,Заказчик_анон,Цена ГК за упак. с НДС_анон,Код_анон_2,Менеджер_анон
0,Производитель_001,Код_001,Препарат_полный_001,Аукцион_001,0.00,М76,NaN,Регион_001,Заказчик_001,59.00,NaN,Менеджер_001
1,Производитель_001,Код_001,Препарат_полный_001,Аукцион_002,0.00,М542,NaN,Регион_002,Заказчик_002,51.96,NaN,Менеджер_002
2,Производитель_001,Код_002,Препарат_полный_002,Аукцион_003,100.00,М49,NaN,Регион_003,Заказчик_003,153.06,NaN,Менеджер_003


In [28]:
result = pd.merge(aviable_leftovers[['Номенклатура.Код', 'Номенклатура', 'Доступно', 'В резерве']], isp_trans, on='Номенклатура.Код', how='outer').reset_index(drop=True)

In [29]:
result = result[result['Количество осталось отгрузить в целом']!=0]

In [30]:
result['Сумма'] = result['Цена ГК за упак. с НДС (нов.)'] * result['Количество осталось отгрузить в целом']

In [31]:
result.columns

Index(['Номенклатура.Код', 'Номенклатура', 'Доступно', 'В резерве',
       'Торговая упаковка.Фирма производитель', 'Наименование полное',
       'Номер аукциона', 'Количество осталось отгрузить в целом',
       'Ответственный', 'Количество упаковок в заявке', 'Субъект',
       'Заказчик в шапке', 'Цена ГК за упак. с НДС (нов.)', 'Код', 'Менеджер',
       'Сумма'],
      dtype='str')

In [32]:
result_columns = ['Номенклатура.Код', 'Номенклатура', 'Доступно', 'В резерве', 'Торговая упаковка.Фирма производитель', 'Наименование полное',
                  'Номер аукциона', 'Количество осталось отгрузить в целом', 'Ответственный', 'Количество упаковок в заявке', 'Субъект',
                  'Заказчик в шапке', 'Цена ГК за упак. с НДС (нов.)', 'Сумма', 'Менеджер']
result = result[result_columns]

#### Объединение с незаконтрактованными

In [ ]:
nezak = pd.read_excel('отчет по складу.xlsx', sheet_name='Данные', usecols=['Номенклатура.Код', 'Наименование', 'Избыток общ', 'Избыток общ, р'])
nezak.head(3)

In [6]:
# Пример данных
demo_overstock_summary = pd.DataFrame([
    {
        "Код_анон": "Код_201",
        "Наименование_анон": "Препарат_201",
        "Избыток общ": 1099665.0,
        "Избыток общ, р": 1873309.71,
    },
    {
        "Код_анон": "Код_202",
        "Наименование_анон": "Препарат_202",
        "Избыток общ": 0.0,
        "Избыток общ, р": 0.0,
    },
    {
        "Код_анон": "Код_203",
        "Наименование_анон": "Препарат_203",
        "Избыток общ": 242583.0,
        "Избыток общ, р": 25137312.90,
    },
])

demo_overstock_summary

,Код_анон,Наименование_анон,Избыток общ,"Избыток общ, р"
0,Код_201,Препарат_201,1099665.00,1873309.71
1,Код_202,Препарат_202,0.00,0.00
2,Код_203,Препарат_203,242583.00,25137312.90


In [34]:
result_nezak = pd.merge(result, nezak[['Номенклатура.Код', 'Избыток общ', 'Избыток общ, р']], on='Номенклатура.Код').reset_index(drop=True)
result_nezak.info()

<class 'pandas.DataFrame'>
RangeIndex: 16101 entries, 0 to 16100
Data columns (total 17 columns):
 #   Column                                 Non-Null Count  Dtype  
---  ------                                 --------------  -----  
 0   Номенклатура.Код                       16101 non-null  str    
 1   Номенклатура                           15273 non-null  str    
 2   Доступно                               15273 non-null  object 
 3   В резерве                              15273 non-null  object 
 4   Торговая упаковка.Фирма производитель  15490 non-null  str    
 5   Наименование полное                    15490 non-null  str    
 6   Номер аукциона                         15490 non-null  str    
 7   Количество осталось отгрузить в целом  15490 non-null  float64
 8   Ответственный                          14666 non-null  str    
 9   Количество упаковок в заявке           1525 non-null   float64
 10  Субъект                                15490 non-null  str    
 11  Заказчик в ша

In [ ]:
result_nezak.head(3)

In [36]:
def extract_vendor_wo_country(text: str) -> str:
    if not isinstance(text, str):
        return None
    # нормализуем пробелы
    s = re.sub(r'\s+', ' ', text).strip()

    # 1) сначала вытащим часть после последней ')'
    pos = s.rfind(')')
    if pos != -1:
        tail = s[pos+1:].strip()
    else:
        tail = s

    # 2) из этого хвоста уберём страну: всё после последнего дефиса
    dash = tail.rfind('-')
    if dash != -1:
        vendor = tail[:dash].strip()
    else:
        vendor = tail

    return vendor

# допустим, исходный столбец называется 'Полное наименование'
result_nezak['Производитель'] = result_nezak['Номенклатура'].apply(extract_vendor_wo_country)

In [37]:
# Заполнение отсутствующих значения
result_nezak['Торговая упаковка.Фирма производитель'] = result_nezak['Торговая упаковка.Фирма производитель'].fillna(result_nezak['Производитель'])
result_nezak['Наименование полное'] = result_nezak['Наименование полное'].fillna(result_nezak['Номенклатура'])

In [38]:
result_nezak.loc[result_nezak['Количество осталось отгрузить в целом'].isna(), 'Менеджер'] = 'Незаконтрактовано'

In [39]:
result_nezak = result_nezak.rename(columns={'Избыток общ':'Избыток тек год', 'Избыток общ, р':'Избыток тек год, р'})

In [40]:
result_nezak_500 = result_nezak[result_nezak['Избыток тек год, р']>=500000]

#### Продажи

In [ ]:
sales = pd.read_excel('продажа.xlsx')[:-1]
sales['Реализация товаров и услуг.Дата'] = pd.to_datetime(sales['Реализация товаров и услуг.Дата'], format='%d.%m.%Y %H:%M:%S')
sales['Характеристика.Серия.Годен до'] = pd.to_datetime(sales['Характеристика.Серия.Годен до'], format='%d.%m.%Y %H:%M:%S')
sales  = sales[sales['Реализация товаров и услуг.Дата'].between(dt_old, dt_new, inclusive='both')]
sales = sales.rename(columns={'Реализация товаров и услуг.Документ исполнение.Заказчик':'Заказчик в шапке'})
sales.head()

In [7]:
# Пример данных
demo_sales_data = pd.DataFrame([
    {
        "Дата_анон": "2026-01-13",
        "Документ_анон": "РН-М0000342",
        "Номенклатура_анон": "Препарат_301",
        "Код_анон": "Код_301",
        "Номер аукциона_анон": "Аукцион_101",
        "Годен до_анон": "2028-09-30",
        "Заказчик_анон": "Заказчик_101",
        "Количество упаковок": 2611.0,
        "Сумма": 104962.20,
        "Сумма НДС": 9542.02,
        "Сумма без НДС": 95420.18,
    },
    {
        "Дата_анон": "2026-01-13",
        "Документ_анон": "РН-Ф0000235",
        "Номенклатура_анон": "Препарат_302",
        "Код_анон": "Код_302",
        "Номер аукциона_анон": "Аукцион_102",
        "Годен до_анон": "2028-08-31",
        "Заказчик_анон": "Заказчик_102",
        "Количество упаковок": 65.0,
        "Сумма": 19370.00,
        "Сумма НДС": 1760.91,
        "Сумма без НДС": 17609.09,
    },
    {
        "Дата_анон": "2026-01-13",
        "Документ_анон": "РН-Ф0000235",
        "Номенклатура_анон": "Препарат_302",
        "Код_анон": "Код_302",
        "Номер аукциона_анон": "Аукцион_102",
        "Годен до_анон": "2028-08-31",
        "Заказчик_анон": "Заказчик_102",
        "Количество упаковок": 135.0,
        "Сумма": 40230.00,
        "Сумма НДС": 3657.27,
        "Сумма без НДС": 36572.73,
    },
    {
        "Дата_анон": "2026-01-13",
        "Документ_анон": "РН-М0000056",
        "Номенклатура_анон": "Препарат_303",
        "Код_анон": "Код_303",
        "Номер аукциона_анон": "Аукцион_103",
        "Годен до_анон": "2028-09-30",
        "Заказчик_анон": "Заказчик_103",
        "Количество упаковок": 2.0,
        "Сумма": 15005.76,
        "Сумма НДС": 1364.16,
        "Сумма без НДС": 13641.60,
    },
    {
        "Дата_анон": "2026-01-13",
        "Документ_анон": "РН-М0000308",
        "Номенклатура_анон": "Препарат_304",
        "Код_анон": "Код_304",
        "Номер аукциона_анон": "Аукцион_104",
        "Годен до_анон": "2030-11-30",
        "Заказчик_анон": "Заказчик_104",
        "Количество упаковок": 300.0,
        "Сумма": 68916.00,
        "Сумма НДС": 6265.09,
        "Сумма без НДС": 62650.91,
    },
])

demo_sales_data

,Дата_анон,Документ_анон,Номенклатура_анон,Код_анон,Номер аукциона_анон,Годен до_анон,Заказчик_анон,Количество упаковок,Сумма,Сумма НДС,Сумма без НДС
0,2026-01-13,РН-М0000342,Препарат_301,Код_301,Аукцион_101,2028-09-30,Заказчик_101,2611.00,104962.20,9542.02,95420.18
1,2026-01-13,РН-Ф0000235,Препарат_302,Код_302,Аукцион_102,2028-08-31,Заказчик_102,65.00,19370.00,1760.91,17609.09
2,2026-01-13,РН-Ф0000235,Препарат_302,Код_302,Аукцион_102,2028-08-31,Заказчик_102,135.00,40230.00,3657.27,36572.73
3,2026-01-13,РН-М0000056,Препарат_303,Код_303,Аукцион_103,2028-09-30,Заказчик_103,2.00,15005.76,1364.16,13641.60
4,2026-01-13,РН-М0000308,Препарат_304,Код_304,Аукцион_104,2030-11-30,Заказчик_104,300.00,68916.00,6265.09,62650.91


In [43]:
# для корректировки даты отчета - datetime(2026, 2, 20).date()
def get_week_data(
    df,
    today,
    date_column='date',
    manual_start=None,
    manual_end=None,
    treat_first_workday_as_monday=True,
):
    # если пришёл date, сделаем Timestamp (pandas-тип)
    today = pd.to_datetime(today)

    # 1. Ручной период
    if manual_start is not None and manual_end is not None:
        print(f'Период: с {manual_start} по {manual_end}')
        start_date = pd.to_datetime(manual_start)
        end_date   = pd.to_datetime(manual_end)
    else:
        wd = today.weekday()  # 0=пн, 1=вт, ...
        print(f"Сегодня: {today:%d.%m.%Y} ({['пн','вт','ср','чт','пт','сб','вс'][wd]})")

        is_effective_monday = (wd == 0) if treat_first_workday_as_monday else (wd == 0)

        if is_effective_monday:
            # понедельник / первый рабочий → прошлая неделя (пн-вс)
            start_date = today - timedelta(days=7)
            end_date   = today - timedelta(days=1)
            period_name = "прошлая неделя"
        else:
            # вт–вс → с понедельника текущей недели по today
            start_date = today - timedelta(days=wd)
            end_date   = today
            period_name = f"с понедельника этой недели по {today:%d.%m.%Y}"

        print(f"Авто-период: с {start_date:%d.%m.%Y} по {end_date:%d.%m.%Y} ({period_name})")

    # 2. Фильтрация: обе стороны datetime64
    col = pd.to_datetime(df[date_column])
    mask = (col >= start_date) & (col <= end_date)
    return df[mask]

In [44]:
sales_week = get_week_data(sales, dt_new, date_column='Реализация товаров и услуг.Дата')

Сегодня: 04.05.2026 (пн)
Авто-период: с 27.04.2026 по 03.05.2026 (прошлая неделя)


In [ ]:
sales_week.head(3)

In [46]:
sales_week_gr = sales_week.groupby(['Номер аукциона', 'Заказчик в шапке', 'Номенклатура.Код']).agg({'Количество упаковок': 'sum'}).reset_index().rename(columns={'Количество упаковок': 'Продано уп за нед'})

In [ ]:
sales_week_gr.head(3)

In [48]:
result_nezak_500 = pd.merge(result_nezak_500, sales_week_gr, on=['Номер аукциона', 'Заказчик в шапке', 'Номенклатура.Код'], how='left').reset_index(drop=True)

In [ ]:
result_nezak_500.head(3)

In [8]:
# Пример итоговой таблицы с данными

demo_full_detail = pd.DataFrame([
    {
        "Код_анон": "Код_401",
        "Номенклатура_анон": "Препарат_401",
        "Доступно": 8594,
        "В резерве": 50,
        "Производитель_анон": "Производитель_401",
        "Наименование полное_анон": "Препарат_полный_401",
        "Номер аукциона_анон": "Аукцион_201",
        "Кол-во осталось отгрузить_анон": 32.0,
        "Ответственный": "М76",
        "Кол-во упаковок в заявке_анон": np.nan,
        "Субъект_анон": "Регион_201",
        "Заказчик_анон": "Заказчик_201",
        "Цена ГК за упак. с НДС_анон": 135.72,
        "Сумма": 4343.04,
        "Менеджер_анон": "Менеджер_201",
        "Избыток тек год": 8544.0,
        "Избыток тек год, р": 1677446.89,
        "Производитель_анон_2": "Производитель_401_МЦ",
        "Продано уп за нед": np.nan,
    },
    {
        "Код_анон": "Код_401",
        "Номенклатура_анон": "Препарат_401",
        "Доступно": 8594,
        "В резерве": 50,
        "Производитель_анон": "Производитель_401",
        "Наименование полное_анон": "Препарат_полный_401",
        "Номер аукциона_анон": "Аукцион_201",
        "Кол-во осталось отгрузить_анон": 18.0,
        "Ответственный": "М76",
        "Кол-во упаковок в заявке_анон": np.nan,
        "Субъект_анон": "Регион_201",
        "Заказчик_анон": "Заказчик_201",
        "Цена ГК за упак. с НДС_анон": 135.72,
        "Сумма": 2442.96,
        "Менеджер_анон": "Менеджер_201",
        "Избыток тек год": 8544.0,
        "Избыток тек год, р": 1677446.89,
        "Производитель_анон_2": "Производитель_401_МЦ",
        "Продано уп за нед": np.nan,
    },
    {
        "Код_анон": "Код_402",
        "Номенклатура_анон": "Препарат_402",
        "Доступно": 242696,
        "В резерве": 0,
        "Производитель_анон": "Производитель_402",
        "Наименование полное_анон": "Препарат_полный_402",
        "Номер аукциона_анон": "Аукцион_202",
        "Кол-во осталось отгрузить_анон": 28.0,
        "Ответственный": "М500",
        "Кол-во упаковок в заявке_анон": np.nan,
        "Субъект_анон": "Регион_202",
        "Заказчик_анон": "Заказчик_202",
        "Цена ГК за упак. с НДС_анон": 130.0,
        "Сумма": 3640.0,
        "Менеджер_анон": "Менеджер_202",
        "Избыток тек год": 242583.0,
        "Избыток тек год, р": 25137312.90,
        "Производитель_анон_2": "Производитель_402",
        "Продано уп за нед": np.nan,
    },
])

demo_full_detail

,Код_анон,Номенклатура_анон,Доступно,В резерве,Производитель_анон,Наименование полное_анон,Номер аукциона_анон,Кол-во осталось отгрузить_анон,Ответственный,Кол-во упаковок в заявке_анон,Субъект_анон,Заказчик_анон,Цена ГК за упак. с НДС_анон,Сумма,Менеджер_анон,Избыток тек год,"Избыток тек год, р",Производитель_анон_2,Продано уп за нед
0,Код_401,Препарат_401,8594,50,Производитель_401,Препарат_полный_401,Аукцион_201,32.00,М76,NaN,Регион_201,Заказчик_201,135.72,4343.04,Менеджер_201,8544.00,1677446.89,Производитель_401_МЦ,NaN
1,Код_401,Препарат_401,8594,50,Производитель_401,Препарат_полный_401,Аукцион_201,18.00,М76,NaN,Регион_201,Заказчик_201,135.72,2442.96,Менеджер_201,8544.00,1677446.89,Производитель_401_МЦ,NaN
2,Код_402,Препарат_402,242696,0,Производитель_402,Препарат_полный_402,Аукцион_202,28.00,М500,NaN,Регион_202,Заказчик_202,130.00,3640.00,Менеджер_202,242583.00,25137312.90,Производитель_402,NaN


#### Выгрузка данных для оверстока

In [50]:
today = datetime.now().strftime('%d.%m.%Y')

In [51]:
result_nezak_500.to_excel(f'Данные для оверсток.xlsx', index=False)